[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/vectors_course.ipynb)

# Vectors full course: practice notebook

One practice question for every teaching chapter of the video, in the same order, and vectors you can turn and
stretch with sliders. Work each question out on paper first, then type your answer into the box and press **check**.
It tells you whether you're right. Press **show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, drag it and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

**Typing answers.** Fractions as `4/5`, roots as `sqrt(14)`, minus as `-`. Angles in degrees as plain numbers
(`38.2`), to the decimals the question asks. For several answers, separate them with commas, in the order the
question asks.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
SUM, ALONG, ACROSS = "#a0c27c", "#6fb6bb", "#c9a0d3"   # the sum or result · the along part · the across part

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
r = sp.Rational
NAMES = {"pi": sp.pi, "sqrt": sp.sqrt}

def read(s):
    """Your typed answer as exact numbers (a list, one for each answer)."""
    s = (s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("°", "").replace("²", "^2"))
    s = re.sub(r"√\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", s).replace("√", "sqrt")   # √14 → sqrt(14); √(…) stays
    return [parse_expr(p, local_dict=NAMES, transformations=RULES) for p in s.split(",") if p.strip()]

def same(a, b, tol=1e-9):
    try: return abs(complex(sp.N(sp.sympify(a) - sp.sympify(b), 30))) < tol
    except TypeError: return False                         # a letter, not a number

# Vectors as plain tuples of numbers: every rule from the video, in one line each.
def add(u, v): return tuple(p + q for p, q in zip(u, v))
def sub(u, v): return tuple(p - q for p, q in zip(u, v))          # head minus tail: sub(head, tail)
def times(k, u): return tuple(k * p for p in u)
def dot(u, v): return sum(p * q for p, q in zip(u, v))
def cross(u, v): return (u[1] * v[2] - u[2] * v[1], u[2] * v[0] - u[0] * v[2], u[0] * v[1] - u[1] * v[0])
def length(u): return math.sqrt(dot(u, u))

def arrow(ax, p, q, col=CHALK, lw=2.8, ls="-", head=22):
    """A vector arrow from point p to point q."""
    ax.annotate("", xy=q, xytext=p, arrowprops=dict(arrowstyle="-|>", color=col, lw=lw, ls=ls, mutation_scale=head))

def column(ax, vals, x, y, col=CHALK, label="", fmt="{:6.2f}"):
    """The arrow's numbers as a column in brackets, beside it: every arrow, two ways (axes coordinates)."""
    rows = "\n".join(fmt.format(v).replace("-", "−") for v in vals)
    ax.text(x, y, rows, color=col, fontsize=16, family="monospace", va="top", ha="left", transform=ax.transAxes,
            bbox=dict(boxstyle="square,pad=0.45", fc=SLATE2, ec=DIM))
    if label: ax.text(x, y + 0.03, label, color=DIM, fontsize=12, va="bottom", ha="left", transform=ax.transAxes)

def flat(ax, xlim, ylim):
    """A flat grid with the two axes through the origin, one unit the same both ways."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect("equal")
    ax.axhline(0, color=DIM, lw=1); ax.axvline(0, color=DIM, lw=1)

def space(ax, lim, view, elev=22):
    """A 3D axes on the slate: no grey panes, a ghost grid, the camera `view` degrees round the vertical."""
    ax.set_facecolor(SLATE)
    for a in (ax.xaxis, ax.yaxis, ax.zaxis):
        a.set_pane_color((0, 0, 0, 0)); a._axinfo["grid"]["color"] = GHOST
    for (lo, hi), setter in zip(lim, (ax.set_xlim, ax.set_ylim, ax.set_zlim)): setter(lo, hi)
    ax.view_init(elev=elev, azim=view)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")

# Each question: the question, the exact answers in order, the working, (for rounded answers) how close counts, and
# (if it has several parts) the move that cracks each part, shown when that part is wrong.
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="420px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            tol = q.get("tol", [1e-9] * len(want))
            ok = [same(g, w, t) for g, w, t in zip(got, want, tol)]
            if all(ok): print("✓ right"); return
            if "moves" not in q: print("✗ not quite: try again, or press show working"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

## 1 · Scalars and vectors

A scalar is a number with a unit: 2 kg, 30 °C, 7 km. A vector has a size and a direction: a move, a velocity, a
push. Walk 3 km east, then 4 km north. You walked 7 km, a scalar. But you are 5 km from home, 53° north of east: that
is a vector, your displacement.

In [ ]:
Q[1] = {"text": "You walk 5 km east, then 12 km north. Give the distance you walked, how far you are from home in a "
                "straight line, and the direction of that straight line as an angle north of east, in degrees to one "
                "decimal place.",
        "answer": [17, 13, 67.4], "tol": [1e-9, 1e-9, 0.051],
        "moves": ["distance is a scalar: just add the two walks.",
                  "the straight line home is the long side of a right triangle: use Pythagoras.",
                  "the angle at home has tan = north ÷ east = 12/5."],
        "working": ["Distance walked: 5 + 12 = 17 km.",
                    "The walks meet at a right angle, so the straight line is the long side of a right triangle:",
                    "5² + 12² = 25 + 144 = 169, and √169 = 13 km.",
                    "Direction: tan θ = 12/5 = 2.4, so θ = tan⁻¹(2.4) ≈ 67.4° north of east."]}
assert 5 + 12 == 17 and math.hypot(5, 12) == 13 and round(math.degrees(math.atan2(12, 5)), 1) == 67.4
practice(1)

## 2 · Addition of vectors

Head to tail, the sum runs from the first tail to the last head: the triangle law. Tail to tail, the sum is the
diagonal of the parallelogram. For two vectors of sizes P and Q with an angle θ between them, tail to tail:
R² = P² + Q² + 2PQ cos θ. Stretch the two arrows and turn the angle between them. The sum is longest at 0° and
shortest at 180°.

In [ ]:
def resultant(stretch_a=3.0, stretch_b=4.0, turn=90):
    t = math.radians(turn)
    a, b = (stretch_a, 0.0), (stretch_b * math.cos(t), stretch_b * math.sin(t))
    s = add(a, b)
    R = math.sqrt(stretch_a ** 2 + stretch_b ** 2 + 2 * stretch_a * stretch_b * math.cos(t))
    assert abs(R - length(s)) < 1e-9                       # the formula and the head-to-tail drawing agree
    fig, ax = plt.subplots(figsize=(11, 5.2))
    ax.fill([0, a[0], s[0], b[0]], [0, a[1], s[1], b[1]], color=SUM, alpha=0.08)
    arrow(ax, a, s, DIM, lw=2, ls="--"); arrow(ax, b, s, DIM, lw=2, ls="--")        # the two copies
    arrow(ax, (0, 0), a, CHALK, lw=3.2); arrow(ax, (0, 0), b, CHALK, lw=3.2)
    arrow(ax, (0, 0), s, SUM, lw=3.6)
    ax.text(a[0] / 2, -0.55, "a", color=CHALK, ha="center", fontsize=15)
    ax.text(b[0] / 2 - 0.35, b[1] / 2 + 0.2, "b", color=CHALK, ha="right", fontsize=15)
    ax.plot(1.1 * np.cos(np.linspace(0, t, 40)), 1.1 * np.sin(np.linspace(0, t, 40)), color=CORAL, lw=2.4)
    flat(ax, (-7, 13), (-1.2, 6.8))
    ax.set_title(f"|a + b| = √({stretch_a:g}² + {stretch_b:g}² + 2·{stretch_a:g}·{stretch_b:g}·cos {turn}°) = {R:.2f}")
    ax.set_xlabel(f"the angle between them (coral): {turn}°   ·   longest at 0°, shortest at 180°")
    column(ax, s, 0.80, 0.90, SUM, "a + b")
    plt.show()

In [ ]:
interact(resultant, stretch_a=FloatSlider(value=3, min=1, max=6, step=0.5, description="stretch a"),
         stretch_b=FloatSlider(value=4, min=1, max=6, step=0.5, description="stretch b"),
         turn=IntSlider(value=90, min=0, max=180, step=15, description="turn (°)"));

In [ ]:
Q[2] = {"text": "Two forces, a of 3 N and b of 5 N, pull on a ring with 60° between them (drawn tail to tail). Give "
                "the size of the resultant a + b, its angle from a in degrees to one decimal place, and the size of "
                "a − b.",
        "answer": [7, 38.2, sp.sqrt(19)], "tol": [1e-9, 0.051, 1e-9],
        "moves": ["R² = P² + Q² + 2PQ cos θ, with θ = 60° and cos 60° = ½.",
                  "tan α = Q sin θ ÷ (P + Q cos θ): the height of b over the length along a.",
                  "a − b = a + (−b). Turning b round makes the angle between a and −b 180° − 60° = 120°."],
        "working": ["R² = 3² + 5² + 2 × 3 × 5 × cos 60° = 9 + 25 + 15 = 49, so R = 7 N.",
                    "tan α = 5 sin 60° ÷ (3 + 5 cos 60°) = 4.330 ÷ 5.5 = 0.787, so α ≈ 38.2° from a.",
                    "a − b: the angle between a and −b is 120°, and cos 120° = −½.",
                    "|a − b|² = 9 + 25 − 15 = 19, so |a − b| = √19 ≈ 4.36 N."]}
_c, _s = math.cos(math.radians(60)), math.sin(math.radians(60))
assert abs(math.sqrt(9 + 25 + 30 * _c) - 7) < 1e-12
assert round(math.degrees(math.atan2(5 * _s, 3 + 5 * _c)), 1) == 38.2
assert abs(length(sub((3, 0), (5 * _c, 5 * _s))) - math.sqrt(19)) < 1e-12
practice(2)

## 3 · Components of a vector

Put the tail at the origin, and every arrow is a part across plus a part up: a = a₁î + a₂ĵ, the same as the column
(a₁, a₂). An arrow of size r at angle θ has parts r cos θ across and r sin θ up. Its magnitude is √(a₁² + a₂²), and
its unit vector â = a ÷ |a| is one unit long, pointing the same way. Stretch the arrow and turn it: the column of
numbers changes with it.

In [ ]:
def turn_and_stretch(stretch=5.0, turn=30):
    t = math.radians(turn)
    a = (stretch * math.cos(t), stretch * math.sin(t))
    u = times(1 / stretch, a)
    assert abs(length(u) - 1) < 1e-12                      # a unit vector is one unit long
    fig, ax = plt.subplots(figsize=(10.5, 4.3))                 # short enough to sit under its heading
    ring = np.linspace(0, 2 * math.pi, 120)
    ax.plot(np.cos(ring), np.sin(ring), color=DIM, lw=1.2, ls=":")
    arrow(ax, (0, 0), (a[0], 0), ALONG, lw=3.2)
    arrow(ax, (a[0], 0), a, ACROSS, lw=3.2)
    arrow(ax, (0, 0), a, CHALK, lw=3.2)
    arrow(ax, (0, 0), u, CORAL, lw=4, head=13)
    arc = np.linspace(0, t, 40) if turn >= 0 else np.linspace(t, 0, 40)
    ax.plot(1.6 * np.cos(arc), 1.6 * np.sin(arc), color=CORAL, lw=2)
    flat(ax, (-7, 14), (-6.5, 6.5))
    ax.set_title(f"a = {a[0]:.2f} î + {a[1]:.2f} ĵ   ·   |a| = {length(a):.2f}   ·   θ = {turn}°".replace("-", "−")
                 .replace("+ −", "− "))
    ax.set_xlabel("across: r cos θ (blue)  ·  up: r sin θ (violet)  ·  unit vector (coral)", fontsize=11)
    column(ax, a, 0.66, 0.90, CHALK, "a")
    column(ax, u, 0.83, 0.90, CORAL, "â = a ÷ |a|")
    plt.show()

In [ ]:
interact(turn_and_stretch, stretch=FloatSlider(value=5, min=1, max=6, step=0.5, description="stretch"),
         turn=IntSlider(value=30, min=-180, max=180, step=15, description="turn (°)"));

In [ ]:
Q[3] = {"text": "a = 8î − 6ĵ. Give |a|, then the two components of the unit vector â. Then b = −5î + 12ĵ: give its "
                "direction, the angle anticlockwise from the positive x axis, in degrees to one decimal place.",
        "answer": [10, r(4, 5), r(-3, 5), 112.6], "tol": [1e-9, 1e-9, 1e-9, 0.051],
        "moves": ["Pythagoras on the components: √(8² + (−6)²).",
                  "divide each component by the magnitude, 10.",
                  "divide each component by the magnitude, 10 (the minus sign stays).",
                  "sketch b first: it points up and to the left, so its angle is in the second quadrant. "
                  "tan⁻¹(12 ÷ −5) on a calculator gives −67.4°; add 180°."],
        "working": ["|a| = √(64 + 36) = √100 = 10.",
                    "â = a ÷ 10 = (8/10)î − (6/10)ĵ = (4/5)î − (3/5)ĵ. Check: 16/25 + 9/25 = 1.",
                    "b = −5î + 12ĵ points up and to the left: the second quadrant.",
                    "tan θ = 12 ÷ −5, and the calculator's −67.4° points down and to the right, the wrong way.",
                    "The true angle is −67.4° + 180° = 112.6°."]}
A3, B3 = (8, -6), (-5, 12)
assert length(A3) == 10 and times(r(1, 10), A3) == (r(4, 5), r(-3, 5))
assert round(math.degrees(math.atan2(B3[1], B3[0])), 1) == 112.6 and round(math.degrees(math.atan(12 / -5)), 1) == -67.4
practice(3)

## 4 · Vectors in three dimensions

A point in space has three coordinates (x, y, z), and its position vector is xî + yĵ + zk̂. Its distance from the
origin is √(x² + y² + z²): Pythagoras across the floor, then up. The vector from P to Q is q − p, head minus tail.
Divide its components by its length and you get its direction cosines, whose squares add to 1. Stretch each
component, and turn the view.

In [ ]:
def arrow_in_space(x=4, y=4, z=7, view=-60):
    v = (x, y, z)
    L = length(v)
    fig = plt.figure(figsize=(11.5, 5.2))
    ax = fig.add_axes([0.0, 0.0, 0.6, 1.0], projection="3d")
    space(ax, ((-2, 8), (-2, 8), (0, 9)), view)
    ax.plot([0, x, x], [0, y, y], [0, 0, z], color=ALONG, lw=1.8, ls="--")     # across the floor, then up
    if L: ax.quiver(0, 0, 0, x, y, z, color=CHALK, lw=3, arrow_length_ratio=0.08)
    ax.scatter([x], [y], [z], color=CORAL, s=40, depthshade=False)
    p = fig.add_axes([0.6, 0.0, 0.4, 1.0]); p.axis("off")
    column(p, v, 0.05, 0.90, CHALK, "a", "{:4.0f}")
    p.text(0.05, 0.50, f"|a| = √({x * x} + {y * y} + {z * z})\n     = {L:.2f}", color=CORAL, fontsize=15,
           family="monospace", va="top")
    if L:
        cos = times(1 / L, v)
        column(p, cos, 0.55, 0.90, SUM, "direction cosines", "{:6.3f}")
        p.text(0.05, 0.22, f"their squares add to {dot(cos, cos):.3f}", color=SUM, fontsize=14)
    ax.set_title("the position vector of (x, y, z)")
    plt.show()

In [ ]:
interact(arrow_in_space, x=IntSlider(value=4, min=0, max=7, description="x"),
         y=IntSlider(value=4, min=0, max=7, description="y"), z=IntSlider(value=7, min=0, max=8, description="z"),
         view=IntSlider(value=-60, min=-180, max=180, step=15, description="turn view"));

In [ ]:
Q[4] = {"text": "P is (2, −1, 4) and Q is (4, 1, 3). Give the length of the vector PQ, then its three direction "
                "cosines l, m and n.",
        "answer": [3, r(2, 3), r(2, 3), r(-1, 3)],
        "moves": ["head minus tail, PQ = q − p, then Pythagoras on its three components.",
                  "divide each component of PQ by its length.", "divide each component of PQ by its length.",
                  "divide each component of PQ by its length (keep the sign)."],
        "working": ["PQ = q − p = (4 − 2, 1 − (−1), 3 − 4) = (2, 2, −1).",
                    "|PQ| = √(4 + 4 + 1) = √9 = 3.",
                    "Direction cosines: (2/3, 2/3, −1/3). Check: 4/9 + 4/9 + 1/9 = 1."]}
PQ4 = sub((4, 1, 3), (2, -1, 4))
assert PQ4 == (2, 2, -1) and length(PQ4) == 3 and sum(c * c for c in times(r(1, 3), PQ4)) == 1
practice(4)

## 5 · Section formula

The point dividing AB internally in the ratio m : n has position vector (n a + m b) ÷ (m + n): each end is
weighted by the other end's number. Externally it is (m b − n a) ÷ (m − n). The midpoint is (a + b) ÷ 2, and the
centroid of a triangle is (a + b + c) ÷ 3.

In [ ]:
def internal(a, b, m, n): return tuple(r(n * p + m * q, m + n) for p, q in zip(a, b))
def external(a, b, m, n): return tuple(r(m * q - n * p, m - n) for p, q in zip(a, b))

A5, B5 = (2, 1, -3), (8, 4, 3)
Q[5] = {"text": "A is (2, 1, −3) and B is (8, 4, 3). Give the point that divides AB internally in the ratio 1 : 2 "
                "(three coordinates), then the point that divides it externally in the ratio 2 : 1 (three more).",
        "answer": list(internal(A5, B5, 1, 2)) + list(external(A5, B5, 2, 1)),
        "moves": ["internal 1 : 2 is (2a + 1b) ÷ 3: each end is weighted by the other end's number."] * 3 +
                 ["external 2 : 1 is (2b − 1a) ÷ (2 − 1)."] * 3,
        "working": ["Internal, m : n = 1 : 2: (2a + b) ÷ 3 = ((4, 2, −6) + (8, 4, 3)) ÷ 3 = (12, 6, −3) ÷ 3 = (4, 2, −1).",
                    "Check: it is a third of the way from A, since (4, 2, −1) − (2, 1, −3) = (2, 1, 2), a third of (6, 3, 6).",
                    "External, m : n = 2 : 1: (2b − a) ÷ 1 = (16, 8, 6) − (2, 1, −3) = (14, 7, 9).",
                    "Check: from A it is twice as far as from B, beyond B."]}
assert Q[5]["answer"] == [4, 2, -1, 14, 7, 9]
P5 = external(A5, B5, 2, 1)
assert length(sub(P5, A5)) == 2 * length(sub(P5, B5))
practice(5)

## 6 · Dot product

a · b = |a||b| cos θ, and in components a₁b₁ + a₂b₂ + a₃b₃. It is the length of b's shadow on a, times |a|. It is
positive when the angle is acute, zero when the two are perpendicular, and negative when the angle is obtuse. The
projection of a on b, the length of a's shadow along b, is a · b ÷ |b|. Turn b and stretch it, and watch the shadow.

In [ ]:
def shadow(stretch=3.0, turn=60):
    t = math.radians(turn)
    a, b = (5.0, 0.0), (stretch * math.cos(t), stretch * math.sin(t))
    ab = dot(a, b)
    assert abs(ab - 5 * stretch * math.cos(t)) < 1e-9      # components and |a||b| cos θ agree
    fig, ax = plt.subplots(figsize=(11, 5.0))
    ax.plot([b[0], b[0]], [0, b[1]], color=DIM, lw=1.6, ls="--")
    ax.plot([0, b[0]], [0, 0], color=ALONG, lw=7, solid_capstyle="butt")          # b's shadow on a
    arrow(ax, (0, 0), a, CHALK, lw=3.2); arrow(ax, (0, 0), b, CHALK, lw=3.2)
    ax.text(5.1, -0.6, "a", color=CHALK, fontsize=15); ax.text(b[0] * 1.08, b[1] * 1.08 + 0.1, "b", color=CHALK, fontsize=15)
    ax.plot(1.0 * np.cos(np.linspace(0, t, 40)), 1.0 * np.sin(np.linspace(0, t, 40)), color=CORAL, lw=2.4)
    flat(ax, (-7, 13), (-1.4, 6.6))
    kind = "zero: perpendicular" if abs(ab) < 1e-9 else ("positive: acute" if ab > 0 else "negative: obtuse")
    ax.set_title(f"a · b = |a||b| cos θ = 5 × {stretch:g} × cos {turn}° = {ab:.2f}".replace("= -", "= −"))
    ax.set_xlabel(f"{kind}   ·   b's shadow on a (blue) = {b[0]:.2f}".replace("= -", "= −"))
    column(ax, (ab,), 0.80, 0.90, CORAL, "a · b")
    plt.show()

In [ ]:
interact(shadow, stretch=FloatSlider(value=3, min=1, max=6, step=0.5, description="stretch b"),
         turn=IntSlider(value=60, min=0, max=180, step=15, description="turn b (°)"));

In [ ]:
Q[6] = {"text": "a = 2î − 2ĵ + k̂ and b = 3î + 4k̂. Give a · b, the angle between a and b in degrees to one decimal "
                "place, and the projection of a on b.",
        "answer": [10, 48.2, 2], "tol": [1e-9, 0.051, 1e-9],
        "moves": ["multiply matching components and add: 2 × 3 + (−2) × 0 + 1 × 4.",
                  "cos θ = a · b ÷ (|a||b|), with |a| = 3 and |b| = 5.",
                  "divide a · b by |b|, the length of the vector you project onto, not by |a|."],
        "working": ["a · b = 6 + 0 + 4 = 10.",
                    "|a| = √(4 + 4 + 1) = 3 and |b| = √(9 + 16) = 5.",
                    "cos θ = 10 ÷ 15 = 2/3, so θ = cos⁻¹(2/3) ≈ 48.2°.",
                    "Projection of a on b = a · b ÷ |b| = 10 ÷ 5 = 2."]}
A6, B6 = (2, -2, 1), (3, 0, 4)
assert dot(A6, B6) == 10 and length(A6) == 3 and length(B6) == 5
assert round(math.degrees(math.acos(10 / 15)), 1) == 48.2 and dot(A6, B6) / length(B6) == 2
practice(6)

## 7 · Cross product

a × b is a vector at right angles to both a and b. Its size, |a||b| sin θ, is the area of the parallelogram on a and
b, and the right-hand rule gives its direction. In components it is the determinant with î, ĵ, k̂ on the top row
(mind the minus sign in the middle). The scalar triple product a · (b × c) is the volume of the box on a, b and c.
Turn b and stretch it: the area grows and shrinks with sin θ, and is zero when the two are parallel.

In [ ]:
def parallelogram(stretch=3.0, turn=60):
    t = math.radians(turn)
    a, b = (4.0, 0.0, 0.0), (stretch * math.cos(t), stretch * math.sin(t), 0.0)
    n = cross(a, b)
    assert abs(n[2] - 4 * stretch * math.sin(t)) < 1e-9 and abs(dot(n, a)) < 1e-9 and abs(dot(n, b)) < 1e-9
    fig, ax = plt.subplots(figsize=(11, 5.0))
    ax.fill([0, a[0], a[0] + b[0], b[0]], [0, 0, b[1], b[1]], color=ACROSS, alpha=0.35, lw=0)
    arrow(ax, (0, 0), a[:2], CHALK, lw=3.2); arrow(ax, (0, 0), b[:2], CHALK, lw=3.2)
    ax.text(4.1, -0.6, "a", color=CHALK, fontsize=15); ax.text(b[0] * 1.08 - 0.2, b[1] * 1.08 + 0.1, "b", color=CHALK, fontsize=15)
    cx, cy = (a[0] + b[0]) / 2, b[1] / 2
    if abs(n[2]) > 1e-9:                                   # ⊙ out of the page, ⊗ into it
        ax.add_patch(plt.Circle((cx, cy), 0.32, fc="none", ec=CORAL, lw=2.4))
        if n[2] > 0: ax.plot([cx], [cy], "o", color=CORAL, ms=6)
        else: ax.plot([cx - 0.22, cx + 0.22], [cy - 0.22, cy + 0.22], color=CORAL, lw=2.4); ax.plot([cx - 0.22, cx + 0.22], [cy + 0.22, cy - 0.22], color=CORAL, lw=2.4)
    flat(ax, (-7, 13), (-6.6, 6.6))
    ax.set_title(f"|a × b| = 4 × {stretch:g} × |sin {turn}°| = {abs(n[2]):.2f}, the shaded area")
    where = ("a and b are parallel: a × b = 0" if abs(n[2]) < 1e-9 else
             "a × b points out of the page, towards you" if n[2] > 0 else "a × b points into the page, away from you")
    ax.set_xlabel(where)
    column(ax, n, 0.80, 0.90, ACROSS, "a × b")
    plt.show()

In [ ]:
interact(parallelogram, stretch=FloatSlider(value=3, min=1, max=6, step=0.5, description="stretch b"),
         turn=IntSlider(value=60, min=-180, max=180, step=15, description="turn b (°)"));

In [ ]:
A7, B7, C7 = (1, 2, -1), (3, 0, 1), (0, 1, 1)
N7 = cross(A7, B7)
Q[7] = {"text": "a = î + 2ĵ − k̂ and b = 3î + k̂. Give a × b (three components), then the area of the triangle with "
                "sides a and b, then the volume of the box on a, b and c = ĵ + k̂.",
        "answer": list(N7) + [sp.sqrt(dot(N7, N7)) / 2, abs(dot(C7, N7))],
        "moves": ["the î part: 2 × 1 − (−1) × 0.", "the ĵ part has the minus in front: −(1 × 1 − (−1) × 3).",
                  "the k̂ part: 1 × 0 − 2 × 3.", "the triangle is half the parallelogram: ½|a × b|.",
                  "the volume is |c · (a × b)|, the scalar triple product."],
        "working": ["a × b = î(2 × 1 − (−1) × 0) − ĵ(1 × 1 − (−1) × 3) + k̂(1 × 0 − 2 × 3) = 2î − 4ĵ − 6k̂.",
                    "Check: (2, −4, −6) · a = 2 − 8 + 6 = 0 and (2, −4, −6) · b = 6 + 0 − 6 = 0.",
                    "|a × b| = √(4 + 16 + 36) = √56, so the triangle's area is ½√56 = √14 ≈ 3.74.",
                    "c · (a × b) = 0 × 2 + 1 × (−4) + 1 × (−6) = −10, so the volume is 10."]}
assert N7 == (2, -4, -6) and dot(N7, A7) == 0 == dot(N7, B7)
assert Q[7]["answer"][3] == sp.sqrt(14) and Q[7]["answer"][4] == 10
practice(7)

## 8 · Lines in space

A line through the point a in the direction b is r = a + λb: each value of λ is one point on the line. In Cartesian
form it is (x − a₁)/b₁ = (y − a₂)/b₂ = (z − a₃)/b₃. The angle between two lines is the angle between their
directions. Two lines that are not parallel and never meet are skew, and the shortest distance between them is
|(a₂ − a₁) · (b₁ × b₂)| ÷ |b₁ × b₂|. Drag λ, and the point walks along the line.

In [ ]:
LA1, LB1 = (1, 1, 0), (1, 2, 2)                          # line 1: r = a₁ + λ b₁
LA2, LB2 = (2, -1, 4), (2, 1, -2)                        # line 2: r = a₂ + μ b₂

def closest():
    """λ and μ of the two nearest points: the joining segment is at right angles to both lines."""
    w = sub(LA2, LA1)
    M = np.array([[dot(LB1, LB1), -dot(LB1, LB2)], [dot(LB1, LB2), -dot(LB2, LB2)]], float)
    return np.linalg.solve(M, [dot(w, LB1), dot(w, LB2)])

def walk(lam=0.0, view=-60):
    pt = add(LA1, times(lam, LB1))
    s, t = closest()
    f1, f2 = add(LA1, times(s, LB1)), add(LA2, times(t, LB2))
    fig = plt.figure(figsize=(11.5, 5.2))
    ax = fig.add_axes([0.0, 0.0, 0.62, 1.0], projection="3d")
    space(ax, ((-3, 6), (-4, 8), (-5, 7)), view)
    k1, k2 = np.array([-2, 3]), np.array([-1.5, 2])                                # kept inside the box
    ax.plot(*[LA1[i] + k1 * LB1[i] for i in range(3)], color=DIM, lw=2)
    ax.plot(*[LA2[i] + k2 * LB2[i] for i in range(3)], color=DIM, lw=2, ls="--")
    ax.plot(*zip(f1, f2), color=ACROSS, lw=3)                                       # the shortest distance
    ax.quiver(0, 0, 0, *LA1, color=CHALK, lw=2.4, arrow_length_ratio=0.15)
    if lam: ax.quiver(*LA1, *times(lam, LB1), color=ALONG, lw=3, arrow_length_ratio=0.08)
    ax.scatter(*[[c] for c in pt], color=CORAL, s=60, depthshade=False)
    p = fig.add_axes([0.62, 0.0, 0.38, 1.0]); p.axis("off")
    column(p, pt, 0.05, 0.90, CORAL, f"r at λ = {lam:g}", "{:5.1f}")
    p.text(0.05, 0.45, f"line 1: r = (1, 1, 0) + λ(1, 2, 2)\nline 2: r = (2, −1, 4) + μ(2, 1, −2)\n\n"
                       f"shortest distance (violet)\n= {length(sub(f2, f1)):.3f}", color=CHALK, fontsize=13, va="top")
    ax.set_title("r = a + λb: blue is λb")
    plt.show()

In [ ]:
interact(walk, lam=FloatSlider(value=0, min=-2, max=3, step=0.5, description="λ"),
         view=IntSlider(value=-60, min=-180, max=180, step=15, description="turn view"));

In [ ]:
N8 = cross(LB1, LB2)
Q[8] = {"text": "Line 1 is r = (î + ĵ) + λ(î + 2ĵ + 2k̂), and line 2 is r = (2î − ĵ + 4k̂) + μ(2î + ĵ − 2k̂). Give the "
                "point on line 1 where λ = 2 (three coordinates), the angle between the two lines in degrees, and the "
                "shortest distance between them.",
        "answer": list(add(LA1, times(2, LB1))) + [90, r(abs(dot(sub(LA2, LA1), N8)), int(length(N8)))],
        "moves": ["a + 2b, component by component."] * 3 +
                 ["the angle between the lines is the angle between b₁ and b₂: start with b₁ · b₂.",
                  "find b₁ × b₂, then |(a₂ − a₁) · (b₁ × b₂)| ÷ |b₁ × b₂|."],
        "working": ["λ = 2: (1, 1, 0) + 2(1, 2, 2) = (3, 5, 4).",
                    "b₁ · b₂ = 2 + 2 − 4 = 0, so the lines are at 90°.",
                    "b₁ × b₂ = (2 × (−2) − 2 × 1, 2 × 2 − 1 × (−2), 1 × 1 − 2 × 2) = (−6, 6, −3), of length 9.",
                    "a₂ − a₁ = (1, −2, 4), and (1, −2, 4) · (−6, 6, −3) = −6 − 12 − 12 = −30.",
                    "Shortest distance = |−30| ÷ 9 = 10/3 ≈ 3.33. Not zero, so the lines are skew: they never meet."]}
assert Q[8]["answer"] == [3, 5, 4, 90, r(10, 3)] and dot(LB1, LB2) == 0 and N8 == (-6, 6, -3)
_s, _t = closest()
assert abs(length(sub(add(LA2, times(_t, LB2)), add(LA1, times(_s, LB1)))) - 10 / 3) < 1e-9   # the brute way agrees
practice(8)

## 9 · Planes

A plane is fixed by one point on it and its normal n, the direction at right angles to it: r · n = d, or
ax + by + cz = d with n = (a, b, c). Through three points, the normal is the cross product of two sides. The
distance of a point (x₁, y₁, z₁) from the plane is |ax₁ + by₁ + cz₁ − d| ÷ |n|.

In [ ]:
A9, B9, C9, P9 = (1, 0, 0), (0, 2, 0), (0, 0, 3), (2, 1, 1)
N9 = cross(sub(B9, A9), sub(C9, A9))
D9 = dot(N9, A9)
Q[9] = {"text": "Find the plane through (1, 0, 0), (0, 2, 0) and (0, 0, 3) as ax + by + cz = d, with d = 6: give a, b, c "
                "and d. Then give the distance of the point P(2, 1, 1) from the plane.",
        "answer": list(N9) + [D9, r(abs(dot(N9, P9) - D9), int(length(N9)))],
        "moves": ["the normal is (B − A) × (C − A) = (−1, 2, 0) × (−1, 0, 3)."] * 3 +
                 ["put any of the three points into 6x + 3y + 2z.",
                  "|6 × 2 + 3 × 1 + 2 × 1 − 6| ÷ |n|, and |n| = √(36 + 9 + 4)."],
        "working": ["Two sides: B − A = (−1, 2, 0) and C − A = (−1, 0, 3).",
                    "Normal: (−1, 2, 0) × (−1, 0, 3) = (2 × 3 − 0 × 0, 0 × (−1) − (−1) × 3, (−1) × 0 − 2 × (−1)) = (6, 3, 2).",
                    "d = n · A = 6, so the plane is 6x + 3y + 2z = 6. (It is x/1 + y/2 + z/3 = 1, the intercept form.)",
                    "Distance of P: |12 + 3 + 2 − 6| ÷ √49 = 11/7 ≈ 1.57."]}
assert Q[9]["answer"] == [6, 3, 2, 6, r(11, 7)] and all(dot(N9, p) == 6 for p in (A9, B9, C9))
practice(9)

## 10 · Vectors in physics

Work is a dot product: W = F · d, in joules when the force is in newtons and the move in metres. Only the part of the
force along the move does any work. Torque is a cross product: τ = r × F, where r runs from the pivot to the point
where the force acts.

In [ ]:
F10, D10 = (3, 2, -1), sub((4, 2, 5), (1, 0, 2))
R10, G10 = (2, 0, 1), (1, 3, 0)
Q[10] = {"text": "A force F = 3î + 2ĵ − k̂ newtons moves an object from (1, 0, 2) to (4, 2, 5), in metres. Give the work "
                 "done, in joules. Then a force F = î + 3ĵ newtons acts at r = 2î + k̂ metres from a pivot: give the "
                 "torque τ = r × F (three components).",
         "answer": [dot(F10, D10)] + list(cross(R10, G10)),
         "moves": ["the move is end minus start, d = (3, 2, 3); then W = F · d."] +
                  ["τ = r × F, in that order: the determinant with r on the middle row and F on the bottom."] * 3,
         "working": ["d = (4, 2, 5) − (1, 0, 2) = (3, 2, 3).",
                     "W = F · d = 3 × 3 + 2 × 2 + (−1) × 3 = 9 + 4 − 3 = 10 J.",
                     "τ = r × F = î(0 × 0 − 1 × 3) − ĵ(2 × 0 − 1 × 1) + k̂(2 × 3 − 0 × 1) = −3î + ĵ + 6k̂ newton metres."]}
assert Q[10]["answer"] == [10, -3, 1, 6] and dot(cross(R10, G10), R10) == 0
practice(10)

## Sources
NCERT Mathematics Class XII, ch. 10 (Vector Algebra) and ch. 11 (Three Dimensional Geometry); Class XI, ch. 11
(Introduction to Three Dimensional Geometry); NCERT Physics Class XI, ch. 3 (Motion in a Plane); OpenStax, Calculus
Volume 3, ch. 2 (Vectors in Space). All retrieved 2026-10-31. Every question is our own.